# 13. Entregable 2: el dashboard y la pestaña «Contexto del problema»

**Autores: Juan David Polo y Gabriella Pallares.**

Este capítulo y los dos siguientes son el **informe del desarrollo del dashboard**. Cada
figura del dashboard aparece aquí con **los mismos datos y cálculos** (se genera con las mismas funciones de
`dashboard/figuras.py`; el dashboard solo les aplica el estilo visual de `dashboard/estilo_figuras.py` y un título que comunica el hallazgo), seguida de su interpretación y de detalles que no caben en el
dashboard: pruebas estadísticas, tablas completas y decisiones de diseño.

| Pestaña del dashboard | Subpestañas | Capítulo del informe |
|---|---|---|
| 1 · Contexto del problema | A · Resumen · B · Datos y partición · C · Objetivo y unidad · D · Método y fuente | 13 (este) |
| 2 · Análisis exploratorio (EDA) | A · Objetivo y calidad · B · Relaciones con el objetivo · C · Tiempo y espacio · D · Auditoría de fuga | 14 |
| 3 · Modelos base (regresión logística) | Desempeño · ROC · Matriz de confusión · Umbral · Calibración y diagnóstico | 15 |

Los cambios metodológicos, de claridad y de presentación de esta versión respecto a la anterior están
listados al final del capítulo (sección 13.8) para que ninguno quede implícito.

> **Guía de lectura.** Cada resultado del dashboard termina aquí con dos bloques: **Interpretación** (qué dice el resultado y cómo leerlo) y **Detalles que no aparecen en el dashboard** (tablas, pruebas, cifras y decisiones que el dashboard omite para no recargarse).

In [1]:
from pathlib import Path
import re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while not (project_root / "_config.yml").exists():
    if project_root.parent == project_root:
        raise FileNotFoundError("No se encontró _config.yml. Ejecuta el notebook dentro de la carpeta del libro.")
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

for _folder in ("tables", "figures"):          # se crean si no existen
    (project_root / _folder).mkdir(exist_ok=True)

from src.project_utils import *

configure_plots()
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)
TABLES = project_root / "tables"

def md_out(text):
    display(Markdown(text))

import plotly.io as pio
from IPython.display import HTML
sys.path.insert(0, str(project_root / "dashboard"))
import figuras as F
import textos as T
_plotly = {"cargado": False}

def mostrar(fig):
    """Muestra una figura Plotly del dashboard (la primera de la página carga plotly.js desde la CDN)."""
    incluir = "cdn" if not _plotly["cargado"] else False
    _plotly["cargado"] = True
    display(HTML(pio.to_html(fig, full_html=False, include_plotlyjs=incluir, config={"displaylogo": False, "responsive": True})))

## Arquitectura y ejecución

```
dashboard/
├── preparar_datos.py          # se ejecuta una vez: lee el CSV y las tablas del Entregable 1, ajusta los
│                              # modelos base, genera las predicciones OOF y guarda datos agregados en datos/
├── datos/                     # CSV/JSON pequeños (lo único que lee el dashboard)
├── figuras.py                 # funciones Plotly compartidas por el dashboard, la vista previa y este libro
├── estilo_figuras.py          # apariencia de las figuras en el dashboard (colores, leyendas, márgenes); no toca datos
├── hallazgos.py               # títulos que comunican el hallazgo de cada gráfico (se calculan desde los artefactos)
├── textos.py                  # textos del dashboard; sus cifras se calculan desde los artefactos
├── app.py                     # aplicación Dash: 3 pestañas, subpestañas y callbacks (expone `server`)
├── exportar_html.py           # genera vista_previa_dashboard.html (estática, sin servidor)
├── verificar_consistencia.py  # comprueba que dashboard, CSV y tablas del libro coinciden
└── assets/estilo.css          # estilos (los comparten la app Dash y la vista previa HTML)
render.yaml                    # despliegue en Render (gunicorn dashboard.app:server)
```

Para usarlo, desde la raíz del proyecto:

```bash
pip install -r requirements.txt
python dashboard/preparar_datos.py     # solo si cambian los datos o los modelos (≈ 1 min)
python dashboard/app.py                # abrir http://127.0.0.1:8050 (puerto configurable con PORT)
gunicorn dashboard.app:server          # producción (Render)
```

**Decisiones de diseño.**

- **Datos precalculados.** El dashboard no lee el CSV de 95 MB ni ajusta modelos al abrirse:
  solo lee agregados pequeños. Así arranca en segundos y funciona en equipos con poca memoria.
- **Una sola fuente para las figuras y las cifras.** `figuras.py` alimenta el dashboard, la vista previa y
  este libro; `textos.py` calcula las cifras de los textos desde los mismos archivos. Las interpretaciones aquí
  corresponden exactamente a lo que se ve en el dashboard, y `verificar_consistencia.py` lo comprueba (sección 13.7).
- **Separación EDA / prueba y origen visible.** El EDA usa solo el entrenamiento; la prueba solo aparece en la pestaña
  de modelos. Cada resultado lleva una insignia: **TRAIN** (entrenamiento), **OOF** (predicciones fuera de pliegue del
  entrenamiento) o **TEST** (prueba cronológica), con los colores de la partición (azul, aqua, naranja).
- **El umbral no se explora con la prueba.** El control de umbral usa solo predicciones OOF; la prueba se evalúa con el umbral 0,50 fijado de antemano (capítulo 15).
- **Color con significado fijo y accesible.** Azul oscuro y azul = entrenamiento y modelo elegido (azul claro = no tardía y modelo aditivo), verde = validación OOF y comprobaciones correctas,
  naranja = solo el conjunto de prueba (TEST), rojo = solo alertas (variables con fuga), gris = referencias (Dummy, gap, cancelados, promedios).
  Los mapas de calor usan una escala secuencial azul (sin rojo).
- **Navegación.** Tres pestañas principales fijas al hacer scroll; el EDA y los modelos se dividen en subpestañas para evitar una
  página excesivamente larga. Diseño responsive.
- **Una sola métrica en el dashboard.** El dashboard muestra únicamente el **ROC-AUC** (la métrica principal) con su IC 95 %, su valor en validación cruzada temporal y la comparación con el Dummy; el control de umbral muestra el porcentaje marcado como tardío y la matriz de confusión. Las demás métricas (Accuracy, Precision, Recall, F1, Balanced Accuracy, Average Precision y Brier) **no cambian** y siguen disponibles en las tablas de resultados y en los capítulos 11 y 15.

## 13.1 Indicadores de la pestaña 1

In [2]:
c = F.contexto()
q = F.calidad()["variables"]
prueba = F.tabla("modelos_prueba.csv").set_index("modelo")
kpis = pd.DataFrame([
    ("Líneas de pedido", es(c["lineas"], 0)),
    ("Pedidos", es(c["pedidos"], 0)),
    ("Clientes", es(c["clientes"], 0)),
    ("Periodo", f"{c['desde']} a {c['hasta']}"),
    ("Prevalencia de entregas tardías (entrenamiento completo, con cancelados)", F.pct(c["tasa_tardia_train"])),
    ("Prevalencia en la población de modelado", F.pct(c["tasa_tardia_modelado"])),
    ("ROC-AUC del modelo base (prueba)", F.num(prueba.loc[F.MODELO_ELEGIDO, "ROC-AUC"])),
], columns=["indicador principal", "valor"])
contexto_extra = pd.DataFrame([
    ("Variables originales", q["originales"]),
    ("Predictoras candidatas tras la auditoría (cap. 9)", f"{q['candidatas_auditoria']}  (de {q['candidatas_originales']} originales sin objetivo, fuga, ids ni datos personales)"),
    ("Líneas de entrenamiento (población de modelado)", f"{es(c['modelado_train'], 0)}  ({es(c['modelado_pedidos_train'], 0)} pedidos)"),
    ("Líneas de prueba (población de modelado)", f"{es(c['modelado_test'], 0)}  ({es(c['modelado_pedidos_test'], 0)} pedidos)"),
    ("Gap temporal entre entrenamiento y prueba", f"{c['gap_dias']} días"),
    ("Unidad de observación", f"línea de pedido (≈ {F.num(T.LINEAS_POR_PEDIDO, 1)} líneas por pedido, misma etiqueta)"),
], columns=["indicador de contexto (añadido)", "valor"])
display(kpis); display(contexto_extra)

,indicador principal,valor
0,Líneas de pedido,180.519
1,Pedidos,65.752
2,Clientes,20.652
3,Periodo,2015-01-01 a 2018-01-31
4,"Prevalencia de entregas tardías (entrenamiento completo, con cance...","54,9 %"
5,Prevalencia en la población de modelado,"57,3 %"
6,ROC-AUC del modelo base (prueba),"0,775"


,indicador de contexto (añadido),valor
0,Variables originales,53
1,Predictoras candidatas tras la auditoría (cap. 9),"19 (de 32 originales sin objetivo, fuga, ids ni datos personales)"
2,Líneas de entrenamiento (población de modelado),137.043 (45.669 pedidos)
3,Líneas de prueba (población de modelado),34.583 (16.850 pedidos)
4,Gap temporal entre entrenamiento y prueba,7 días
5,Unidad de observación,"línea de pedido (≈ 3,0 líneas por pedido, misma etiqueta)"


In [3]:
ref = F.tabla("referencia_empirica.csv").iloc[0]
md_out(f"**Interpretación.** El volumen ({es(c['lineas'], 0)} líneas en {es(c['pedidos'], 0)} pedidos de {es(c['clientes'], 0)} clientes) supera "
       "holgadamente el mínimo de 20.000 observaciones, pero el **tamaño efectivo** está más cerca del número de pedidos: todas las líneas de "
       "un pedido comparten la etiqueta (sección 13.2). Más de la mitad de las líneas llega tarde "
       f"({F.pct(c['tasa_tardia_train'])} en el entrenamiento completo y {F.pct(c['tasa_tardia_modelado'])} en la población de modelado, sin cancelados), "
       "así que el problema **no es de clase rara**: es un problema de **ordenar bien** qué líneas tienen más riesgo, lo que justifica el ROC-AUC como "
       f"métrica principal (sección 13.5). El último indicador resume el resultado: el modelo base alcanza un ROC-AUC de {F.num(prueba.loc[F.MODELO_ELEGIDO, 'ROC-AUC'])} "
       f"en la prueba, prácticamente igual a la referencia empírica sin fuga basada en modo × jornada ({F.num(ref['ROC-AUC'])}). "
       "Las tarjetas de contexto añadidas resumen en un vistazo la escala del problema (53 variables originales, "
       f"{q['candidatas_auditoria']} candidatas tras la auditoría), el tamaño de cada conjunto y el gap temporal que protege contra la fuga entre entrenamiento y prueba.")

**Interpretación.** El volumen (180.519 líneas en 65.752 pedidos de 20.652 clientes) supera holgadamente el mínimo de 20.000 observaciones, pero el **tamaño efectivo** está más cerca del número de pedidos: todas las líneas de un pedido comparten la etiqueta (sección 13.2). Más de la mitad de las líneas llega tarde (54,9 % en el entrenamiento completo y 57,3 % en la población de modelado, sin cancelados), así que el problema **no es de clase rara**: es un problema de **ordenar bien** qué líneas tienen más riesgo, lo que justifica el ROC-AUC como métrica principal (sección 13.5). El último indicador resume el resultado: el modelo base alcanza un ROC-AUC de 0,775 en la prueba, prácticamente igual a la referencia empírica sin fuga basada en modo × jornada (0,775). Las tarjetas de contexto añadidas resumen en un vistazo la escala del problema (53 variables originales, 19 candidatas tras la auditoría), el tamaño de cada conjunto y el gap temporal que protege contra la fuga entre entrenamiento y prueba.

In [4]:
md_out(f"""**Detalles que no aparecen en el dashboard.**

- El indicador principal del dashboard es la prevalencia del **entrenamiento completo** ({F.pct(c['tasa_tardia_train'])}). Aquí se separa de la prevalencia en la **población de modelado**
  ({F.pct(c['tasa_tardia_modelado'])}), que excluye las {es(c['train_lineas'] - c['modelado_train'], 0)} líneas canceladas o sospechosas de fraude del entrenamiento; la diferencia entre ambas cifras proviene solo de esa exclusión.
- Tamaño de cada bloque (líneas): entrenamiento {es(c['train_lineas'], 0)}, gap {es(c['gap_lineas'], 0)}, prueba {es(c['test_lineas'], 0)}; en la población de modelado, {es(c['modelado_train'], 0)} y {es(c['modelado_test'], 0)}.
- La tabla de indicadores de contexto (variables originales, candidatas, gap) y el número de países de destino ({c['paises_destino']}) solo se detallan en este capítulo.
""")

**Detalles que no aparecen en el dashboard.**

- El indicador principal del dashboard es la prevalencia del **entrenamiento completo** (54,9 %). Aquí se separa de la prevalencia en la **población de modelado**
  (57,3 %), que excluye las 6.032 líneas canceladas o sospechosas de fraude del entrenamiento; la diferencia entre ambas cifras proviene solo de esa exclusión.
- Tamaño de cada bloque (líneas): entrenamiento 143.075, gap 1.222, prueba 36.222; en la población de modelado, 137.043 y 34.583.
- La tabla de indicadores de contexto (variables originales, candidatas, gap) y el número de países de destino (164) solo se detallan en este capítulo.


## 13.2 Pedido vs línea de pedido: el nivel de análisis

**Decisión oficial del Entregable 1.** La pregunta de investigación habla de «un pedido», pero el Entregable 1 define explícitamente
la **unidad de observación como una línea (ítem) de un pedido** (capítulos 1 y 2), y todo el libro (partición, validación cruzada,
modelo, métricas) trabaja a ese nivel. Por eso el dashboard **conserva la metodología** y corrige los textos para no hablar como si
cada fila fuera un pedido independiente. No se cambió el modelo ni se pasó a una fila por `order_id`: eso alteraría el tamaño de
muestra, los pliegues y todas las métricas del Entregable 1, y habría que documentarlo y regenerarlo todo. Dashboard y Jupyter Book
usan el **mismo nivel de análisis**: la línea de pedido.

In [5]:
tam = pd.read_csv(TABLES / "tamano_muestra.csv").set_index("medida")["valor"]
unidad = pd.DataFrame([
    ("Líneas de entrenamiento (con cancelados)", tam["Líneas (filas) de entrenamiento"]),
    ("Pedidos distintos en entrenamiento", tam["Pedidos distintos"]),
    ("Líneas por pedido (media; máx.)", tam["Líneas por pedido (media; máx.)"]),
    ("Pedidos con la misma etiqueta en todas sus líneas", tam["Pedidos con la misma etiqueta en todas sus líneas"]),
    ("Líneas por pedido en la población de modelado: entrenamiento", F.num(c["modelado_train"] / c["modelado_pedidos_train"], 2)),
    ("Líneas por pedido en la población de modelado: prueba", F.num(c["modelado_test"] / c["modelado_pedidos_test"], 2)),
    ("Pedidos compartidos entre entrenamiento y prueba (recomprobado en preparar_datos.py)", F.calidad()["integridad"]["pedidos_compartidos_train_test"]),
], columns=["medida", "valor"])
unidad

,medida,valor
0,Líneas de entrenamiento (con cancelados),143.075
1,Pedidos distintos en entrenamiento,47.719
2,Líneas por pedido (media; máx.),"3,00; 5"
3,Pedidos con la misma etiqueta en todas sus líneas,"100,0 %"
4,Líneas por pedido en la población de modelado: entrenamiento,"3,00"
5,Líneas por pedido en la población de modelado: prueba,"2,05"
6,Pedidos compartidos entre entrenamiento y prueba (recomprobado en ...,0


**Consecuencias de trabajar a nivel de línea** (y cómo se manejan):

1. **La etiqueta es la misma para todas las líneas de un pedido** (100 % de los pedidos), de modo que la respuesta a «¿llegará tarde el pedido?» es la
   predicción de sus líneas. El modelo usa solo modo de envío y jornada, que son iguales en todas las líneas de un pedido.
2. **Ningún pedido queda dividido entre entrenamiento y prueba**: la partición es cronológica por fecha del pedido y el código vuelve a
   comprobarlo (pedidos compartidos = 0).
3. **Los intervalos de confianza se obtienen remuestreando pedidos** (`order_id`), no líneas: remuestrear líneas trataría como independientes
   observaciones que comparten etiqueta y daría intervalos demasiado estrechos.
4. **Las métricas pesan cada línea**, así que un pedido con más líneas pesa más en Accuracy, Precision, Recall, etc. Desde octubre de 2017 cada pedido
   tiene una sola línea (capítulo 7), por lo que la prueba mezcla dos regímenes; el desempeño por periodo (capítulo 15) muestra que no cambia.
5. **El tamaño efectivo** de muestra está más cerca del número de pedidos que del de líneas.
Un análisis a nivel de pedido (una fila por `order_id`) queda como extensión posible, no como parte de este entregable.

In [6]:
md_out(f"""**Interpretación.** El dashboard resume la unidad como «línea de pedido, ≈ {F.num(T.LINEAS_POR_PEDIDO, 1)} líneas por pedido, misma etiqueta». La consecuencia práctica es que los
{es(c['modelado_train'], 0)} registros de entrenamiento provienen de solo {es(c['modelado_pedidos_train'], 0)} pedidos: el tamaño efectivo de la muestra está más cerca del número de pedidos que del de líneas,
y por eso los intervalos de confianza se obtienen remuestreando pedidos.

**Detalles que no aparecen en el dashboard.** La tabla de arriba (pedidos con la misma etiqueta en todas sus líneas, líneas por pedido en cada conjunto, pedidos compartidos entre entrenamiento y prueba = 0)
y las cinco consecuencias de trabajar a nivel de línea.""")

**Interpretación.** El dashboard resume la unidad como «línea de pedido, ≈ 3,0 líneas por pedido, misma etiqueta». La consecuencia práctica es que los
137.043 registros de entrenamiento provienen de solo 45.669 pedidos: el tamaño efectivo de la muestra está más cerca del número de pedidos que del de líneas,
y por eso los intervalos de confianza se obtienen remuestreando pedidos.

**Detalles que no aparecen en el dashboard.** La tabla de arriba (pedidos con la misma etiqueta en todas sus líneas, líneas por pedido en cada conjunto, pedidos compartidos entre entrenamiento y prueba = 0)
y las cinco consecuencias de trabajar a nivel de línea.

## 13.3 Problema, importancia y pregunta

El texto que muestra la pestaña:

In [7]:
md_out("> " + T.PROBLEMA + "\n\n> **Pregunta:** " + T.PREGUNTA + "\n\n> **Unidad:** " + T.UNIDAD)

> DataCo Global vende artículos deportivos, ropa y electrónica a 164 países. Una parte importante de sus entregas llega después de la fecha prometida. Las entregas tardías deterioran la satisfacción y la retención de clientes, generan reclamos, devoluciones y penalizaciones, y obligan a reprogramar transporte e inventario con sobrecostos. Predecir el riesgo de retraso **en el momento de la compra** permite intervenir antes del despacho (cambiar el modo de envío, priorizar pedidos o ajustar la fecha prometida) en lugar de reaccionar cuando el daño ya ocurrió.

> **Pregunta:** ¿Es posible predecir, al crear un pedido y solo con información disponible en ese instante, si su entrega llegará tarde respecto al plazo prometido?

> **Unidad:** **Unidad de observación: la línea de pedido** (un producto dentro de un pedido), no el pedido completo. Un pedido puede tener varias líneas (3,0 en promedio en el entrenamiento) y **todas las líneas de un pedido comparten la misma etiqueta**, así que la respuesta para un pedido es la de sus líneas. El modelado, la validación y las métricas son a nivel de línea; la partición cronológica deja cada pedido completo en un solo conjunto (pedidos compartidos entre entrenamiento y prueba: 0) y los intervalos de confianza se obtienen remuestreando **pedidos** (`order_id`), no líneas.

**Por qué es relevante usar ML con esta información.** La empresa ya registra, al momento de la
compra, el modo de envío, la promesa de días, el producto, el cliente, el destino y la hora. Si
con esos datos se puede **anticipar** el retraso, la intervención (cambiar el modo de envío,
priorizar el despacho, ajustar la fecha prometida o avisar al cliente) ocurre **antes** de que el
retraso suceda y su costo se materialice. Además, identificar qué condiciones generan los retrasos
permite corregir **políticas** (por ejemplo, promesas poco realistas), no solo casos individuales.

In [8]:
md_out(f"""**Interpretación.** El problema se plantea como anticipar, al crear el pedido, si la línea llegará después de la fecha prometida. Con una prevalencia de {F.pct(c['tasa_tardia_train'])} en el entrenamiento,
no es un problema de clase rara: lo útil es **ordenar** las líneas por riesgo, y por eso el dashboard presenta la tasa como indicador principal y la pregunta de investigación como guía de todo lo demás.

**Detalles que no aparecen en el dashboard.** El texto completo del problema y de la unidad (citas de arriba), el argumento de por qué usar aprendizaje automático con la información disponible al comprar, y una limitación:
el beneficio de intervenir antes (cambiar el modo de envío, priorizar el despacho, avisar al cliente) es una **hipótesis operativa**; en este entregable no se estimó el costo de un retraso ni el de una intervención.""")

**Interpretación.** El problema se plantea como anticipar, al crear el pedido, si la línea llegará después de la fecha prometida. Con una prevalencia de 54,9 % en el entrenamiento,
no es un problema de clase rara: lo útil es **ordenar** las líneas por riesgo, y por eso el dashboard presenta la tasa como indicador principal y la pregunta de investigación como guía de todo lo demás.

**Detalles que no aparecen en el dashboard.** El texto completo del problema y de la unidad (citas de arriba), el argumento de por qué usar aprendizaje automático con la información disponible al comprar, y una limitación:
el beneficio de intervenir antes (cambiar el modo de envío, priorizar el despacho, avisar al cliente) es una **hipótesis operativa**; en este entregable no se estimó el costo de un retraso ni el de una intervención.

## 13.4 Definición de la variable objetivo (corregida)

La versión anterior simplificaba: «`Late_delivery_risk` = 1 si días reales > días programados; 0 si llega temprano o a tiempo». Eso **no es exacto en
el dataset original**: los pedidos cancelados o sospechosos de fraude tienen etiqueta 0 aunque nunca se entregaron, es decir, no son entregas puntuales.

In [9]:
md_out("> " + T.OBJETIVO)
d = F.tabla("objetivo_por_estado.csv").pivot(index="delivery_status", columns="late_delivery_risk", values="lineas").fillna(0).astype(int)
d.columns = ["Late_delivery_risk = 0", "Late_delivery_risk = 1"]
display(d)
inc = F.calidad()["inconsistencias"]
md_out(f"En el entrenamiento hay **{es(inc['lineas_tardias_por_dias_con_etiqueta_0'], 0)} líneas** con días reales > días programados y etiqueta 0: todas son «Shipping canceled» "
       "(pedidos CANCELED o SUSPECTED_FRAUD, capítulo 2). Por eso la definición válida es la de la **población de modelado**: tras excluir los cancelados/fraude, "
       "1 = entrega tardía y 0 = entrega no tardía (adelantada o a tiempo). La figura de la pestaña EDA muestra los cancelados en gris, separados del 0 «no tardía».")

> En la población de modelado, tras excluir los pedidos cancelados o sospechosos de fraude, `Late_delivery_risk` = 1 identifica una entrega tardía (días reales > días programados) y 0 una entrega no tardía (adelantada o a tiempo). En el dataset original, los pedidos cancelados o sospechosos de fraude tienen etiqueta 0 aunque **no** representan una entrega puntual (nunca se entregaron): por eso se excluyen (6.032 líneas del entrenamiento) y no se interpretan como entregas a tiempo.

,Late_delivery_risk = 0,Late_delivery_risk = 1
delivery_status,,
Advance shipping,32954,0
Late delivery,0,78549
Shipping canceled,6032,0
Shipping on time,25540,0


En el entrenamiento hay **3.362 líneas** con días reales > días programados y etiqueta 0: todas son «Shipping canceled» (pedidos CANCELED o SUSPECTED_FRAUD, capítulo 2). Por eso la definición válida es la de la **población de modelado**: tras excluir los cancelados/fraude, 1 = entrega tardía y 0 = entrega no tardía (adelantada o a tiempo). La figura de la pestaña EDA muestra los cancelados en gris, separados del 0 «no tardía».

In [10]:
md_out(f"""**Interpretación.** En el dashboard la variable objetivo se muestra con sus dos clases y una alerta: las etiquetas 0 de los cancelados y sospechosos de fraude **no son entregas a tiempo**. En el entrenamiento hay
{es(inc['lineas_tardias_por_dias_con_etiqueta_0'], 0)} líneas con días reales > programados y etiqueta 0, todas «Shipping canceled»; por eso la definición válida del 0 («no tardía») rige solo para la población de modelado,
que excluye {es(c['train_lineas'] - c['modelado_train'], 0)} líneas del entrenamiento.

**Detalles que no aparecen en el dashboard.** La tabla cruzada de estado de entrega × etiqueta (arriba), el conteo exacto de líneas con etiqueta inconsistente y la explicación de por qué la definición simplificada del Entregable 1
(«0 = temprano o a tiempo») no es exacta en el dataset original.""")

**Interpretación.** En el dashboard la variable objetivo se muestra con sus dos clases y una alerta: las etiquetas 0 de los cancelados y sospechosos de fraude **no son entregas a tiempo**. En el entrenamiento hay
3.362 líneas con días reales > programados y etiqueta 0, todas «Shipping canceled»; por eso la definición válida del 0 («no tardía») rige solo para la población de modelado,
que excluye 6.032 líneas del entrenamiento.

**Detalles que no aparecen en el dashboard.** La tabla cruzada de estado de entrega × etiqueta (arriba), el conteo exacto de líneas con etiqueta inconsistente y la explicación de por qué la definición simplificada del Entregable 1
(«0 = temprano o a tiempo») no es exacta en el dataset original.

## 13.5 Métrica elegida, métricas complementarias y lenguaje de referencia

In [11]:
md_out(T.METRICA + "\n\n" + T.SIN_FUGA)

**Métrica principal: ROC-AUC.** Mide la capacidad de ordenar las líneas por riesgo (el uso previsto), no depende del umbral de decisión y las clases están casi balanceadas (≈ 57 % tardías en la población de modelado). Sirve también para detectar fuga: el Dummy vale 0,5, la referencia empírica sin fuga ≈ 0,775 y con fuga se llega a ≈ 0,975. Accuracy, Precision, Recall, F1, Balanced Accuracy, Average Precision y Brier son métricas complementarias; las que dependen del umbral se reportan con el umbral 0,50 fijado de antemano.

**Versión sin fuga.** Se excluyen las variables que solo se conocen después del despacho: días reales de envío, estado de la entrega, estado del pedido y fecha de despacho.

La justificación completa de la métrica principal (sensibilidad al umbral, comparación con accuracy y F1) está en el capítulo 12.
**Actualización para el Entregable 2:** el capítulo 12 descartó *Precision* como métrica de **reporte principal**; en esta versión se **añade como
métrica complementaria** (junto con *Balanced Accuracy*) porque ayuda a leer el efecto del umbral y a interpretar la curva Precision–Recall. El ROC-AUC sigue
siendo la métrica principal y el Accuracy nunca lo es.

In [12]:
glosario = pd.DataFrame([(k, v) for k, v in T.TOOLTIPS.items()], columns=["métrica", "qué mide"])
glosario

,métrica,qué mide
0,ROC-AUC,Probabilidad de que una línea tardía reciba un puntaje mayor que u...
1,Accuracy,Proporción de aciertos con el umbral fijado. Engañosa cuando una c...
2,Precision,"De las líneas marcadas como tardías, ¿qué fracción realmente lo es..."
3,Recall,"De las líneas realmente tardías, ¿qué fracción se detecta? TP / (T..."
4,F1,Media armónica de Precision y Recall. Depende del umbral y de la p...
5,Balanced Accuracy,"Promedio de Recall y especificidad: el Dummy vale 0,5 sin importar..."
6,Average Precision,Área bajo la curva Precision–Recall (resume Precision a distintos ...
7,Brier,Error cuadrático medio de las probabilidades (menor es mejor). Mid...
8,ROC-AUC CV temporal,ROC-AUC medio de la validación cruzada temporal del entrenamiento ...


**Precision** = TP / (TP + FP): de las líneas que el modelo marca como tardías, qué fracción realmente lo es. Complementa al **Recall** (TP / (TP + FN)).
Como la clase positiva (tardía) es la mayoritaria, un clasificador que marca todo ya tiene Precision igual a la prevalencia; por eso se interpreta junto con
el Recall y con la curva Precision–Recall, no sola. **Balanced Accuracy** = (Recall + especificidad) / 2: vale 0,5 para cualquier clasificador sin información,
sin importar la prevalencia, lo que la hace más honesta que el Accuracy. Ambas dependen del umbral (0,50, fijado de antemano). Sus IC 95 % se calculan con el mismo bootstrap por pedido.

**Cambio de lenguaje: «referencia empírica», no «techo».** Las versiones anteriores hablaban de «techo sin fuga», «techo alcanzable» y «no queda más señal por extraer».
Eso afirma más de lo demostrado. Lo que se observa es que la **tabla de tasas por modo × jornada** (calculada en entrenamiento y evaluada en la prueba) alcanza un ROC-AUC
prácticamente igual al de la regresión logística con la misma interacción; con las variables y relaciones evaluadas no se encontró señal adicional, pero no se demuestra que ningún
otro modelo pueda superarla. Por eso el dashboard habla de **referencia empírica sin fuga** o **benchmark empírico observado con la información disponible**. (Los capítulos 9–12 del Entregable 1 conservan su redacción original;
los archivos `techo_sin_fuga.csv` siguen con ese nombre para no romper el libro.)

In [13]:
pr_ = F.tabla("modelos_prueba.csv").set_index("modelo")
dm, el_ = pr_.loc[F.MODELO_DUMMY], pr_.loc[F.MODELO_ELEGIDO]
md_out(f"""**Interpretación.** El dashboard presenta como única métrica el **ROC-AUC**: ordena las líneas por riesgo, no depende del umbral y no se infla por la clase mayoritaria. El Dummy lo muestra: con solo marcar todo como tardío alcanza
un Accuracy de {F.num(dm['Accuracy'])} pero un ROC-AUC de {F.num(dm['ROC-AUC'], 1)}; el modelo elegido llega a {F.num(el_['ROC-AUC'])}. Por eso Accuracy, F1 y Recall no son la métrica principal.

**Detalles que no aparecen en el dashboard.** Las métricas complementarias (Accuracy, Precision, Recall, F1, Balanced Accuracy, Average Precision y Brier, con sus IC 95 %) **se conservan en este informe** (capítulo 15) pero se retiraron del dashboard
para no competir con la métrica principal. También quedan aquí el glosario de métricas, la justificación completa de la elección (capítulo 12) y el cambio de lenguaje «referencia empírica» en lugar de «techo».""")

**Interpretación.** El dashboard presenta como única métrica el **ROC-AUC**: ordena las líneas por riesgo, no depende del umbral y no se infla por la clase mayoritaria. El Dummy lo muestra: con solo marcar todo como tardío alcanza
un Accuracy de 0,573 pero un ROC-AUC de 0,5; el modelo elegido llega a 0,775. Por eso Accuracy, F1 y Recall no son la métrica principal.

**Detalles que no aparecen en el dashboard.** Las métricas complementarias (Accuracy, Precision, Recall, F1, Balanced Accuracy, Average Precision y Brier, con sus IC 95 %) **se conservan en este informe** (capítulo 15) pero se retiraron del dashboard
para no competir con la métrica principal. También quedan aquí el glosario de métricas, la justificación completa de la elección (capítulo 12) y el cambio de lenguaje «referencia empírica» en lugar de «techo».

## 13.6 Partición cronológica

In [14]:
mostrar(F.fig_particion())
v = F.tabla("volumen_semanal.csv")
res = v.groupby("conjunto")["lineas"].sum()
md_out(f"Líneas por conjunto: " + ", ".join(f"{k}: {v_:,}".replace(",", ".") for k, v_ in res.items()) + ".")

Líneas por conjunto: Entrenamiento: 143.075, Gap (descartado): 1.222, Prueba: 36.222.

**Interpretación.** El volumen semanal es casi constante (≈ 1.200 líneas por semana) hasta
septiembre de 2017. Desde octubre de 2017 cae a menos de la mitad: cada pedido pasa a tener una
sola línea (cambio de régimen descrito en el capítulo 7). Ese periodo queda dentro de la prueba,
lo que convierte la evaluación en una prueba exigente de **generalización a un régimen nuevo**.
La franja gris (gap de 7 días) separa entrenamiento y prueba porque la etiqueta de un pedido se
conoce hasta 6 días después de comprarlo (`days_shipping_real` ≤ 6).

**Detalles que no aparecen en el dashboard.**

- La partición se calcula una vez y se guarda en `tables/particion_pedidos.csv`; el dashboard la
  lee ya hecha.
- La figura usa **solo el volumen** (sin etiquetas) en la prueba: mostrar la tasa de retraso de la
  prueba en la pestaña de contexto sería mirar el resultado antes de evaluar.
- La población de modelado excluye 6.032 líneas canceladas o sospechosas de fraude del
  entrenamiento (y las equivalentes de la prueba); el EDA las conserva para describirlas.

## 13.7 Verificación de consistencia dashboard ↔ libro

`dashboard/verificar_consistencia.py` compara las cifras del dashboard con las tablas del libro (capítulos 2, 9 y 11) y comprueba las reglas metodológicas (umbral con OOF, pedidos compartidos = 0, calibración sin partir empates):

In [15]:
sys.path.insert(0, str(project_root / "dashboard"))
from verificar_consistencia import verificar
vc = verificar()
display(vc)
assert (vc["resultado"] == "OK").all(), "Hay cifras que no coinciden entre el dashboard y el libro"

,comprobación,resultado,detalle
0,Métricas en la prueba de «Logística modo × jornada» = capítulo 11,OK,
1,Métricas en la prueba de «Dummy — clase mayoritaria / prior» = cap...,OK,
2,"Referencia empírica (prueba) = capítulo 11 (Accuracy, Recall, F1, ...",OK,
3,IC 95 % de «Regresión logística» = capítulo 11 (6 métricas),OK,
4,IC 95 % de «Dummy» = capítulo 11 (6 métricas),OK,
5,ROC-AUC CV temporal del modelo elegido = capítulo 11,OK,
6,"Estimaciones de modelos_ic.csv = modelos_prueba.csv (8 métricas, i...",OK,
7,Precision y Balanced Accuracy de la tabla = recalculadas desde pre...,OK,
8,Líneas y pedidos de entrenamiento = capítulo 2 (tamano_muestra.csv),OK,
9,Variables originales = 53 (diccionario) y candidatas = auditoría d...,OK,


## 13.8 Registro de cambios de esta versión

| Tema | Antes | Ahora |
|---|---|---|
| Unidad de análisis | «pedido» en la pregunta, «línea» en la unidad | Explícito: nivel de línea; IC por `order_id`; sin cambiar el modelo (13.2) |
| Definición del objetivo | «0 = temprano o a tiempo» | 0 = no tardía **en la población de modelado**; cancelados/fraude excluidos y mostrados aparte (13.4) |
| «Techo sin fuga» | afirmación absoluta | «Referencia empírica sin fuga»; sin afirmar un máximo demostrado (13.5) |
| Métricas | Accuracy, Recall, F1, ROC-AUC, AP, Brier | + **Precision** y **Balanced Accuracy** (con IC 95 % por bootstrap de pedidos) |
| IC 95 % | bootstrap con 500 remuestreos (ROC-AUC 0,766–0,782) | 1.000 remuestreos, como en el capítulo 11: los IC coinciden exactamente con `tables/intervalos_bootstrap_*.csv` (ROC-AUC 0,767–0,783) |
| Dummy | «Dummy (prior)» | «Dummy — clase mayoritaria / prior», con explicación de su Recall = 1 y su F1 alto |
| Umbral | el deslizador usaba las predicciones de la **prueba** | usa predicciones **OOF del entrenamiento**; la prueba se evalúa aparte con el umbral 0,50 fijado de antemano |
| Calibración | deciles por rango (partían empates) | un punto por probabilidad única (modo × jornada) o deciles que preservan empates (aditivo); n en el hover |
| Curva Precision–Recall | no existía | calculada (capítulo 15 y `figuras.fig_pr`), pero el dashboard muestra solo la curva ROC |
| Pestaña de contexto | una sola página con 12 indicadores y 5 tarjetas de texto, mezclando entrenamiento y prueba | 4 subpestañas: Resumen (4 indicadores + problema y pregunta), Datos y partición (explica en lenguaje sencillo qué es TRAIN y TEST, gráfico y tamaños), Objetivo y unidad, Método y fuente; el ROC-AUC de prueba se quitó del contexto (está en Modelos) |
| Diseño visual y métrica | estética de documento: tarjetas en todo, párrafos largos, siete métricas, rojo para la clase «tardía» | estética de aplicación de analítica: KPI principal, títulos con hallazgos, texto corto con detalle desplegable, gráficos sin tarjeta, **solo ROC-AUC**, rojo solo para alertas, naranja solo para TEST, autores en el encabezado |
| Organización | EDA y modelos en páginas largas | subpestañas internas, pestañas fijas, insignias TRAIN / OOF / TEST, tarjetas de calidad |
| Lenguaje de resultados | «no hay patrón geográfico», «no hay tendencia», «no favorece ni perjudica», «aprendió exactamente las reglas», «más datos no mejorarían» | formulaciones acotadas a lo que se midió (capítulos 14 y 15) |
| Cifras de los textos | escritas a mano | calculadas desde los artefactos (`textos.py`) |
| Despliegue | solo ejecución local en 127.0.0.1 | `render.yaml`, `gunicorn dashboard.app:server`, host 0.0.0.0 y puerto por variable `PORT` |
| Informe de desarrollo | Interpretaciones y detalles mezclados o ausentes en algunos resultados | Cada resultado del dashboard tiene dos bloques en los capítulos 13–15: **Interpretación** y **Detalles que no aparecen en el dashboard** |

Las métricas previas (ROC-AUC, Accuracy, Recall, F1, AP, Brier) y los resultados de los capítulos 1 a 12 **no cambian**; `verificar_consistencia.py` lo comprueba contra las tablas del capítulo 11.